# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/josh777-ops/Fly-Rank-AI/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding A: ML Appendix — "What Predicts Health?" (Random Forest feature importance)

The paper reports Average Position (43%), Impressions (32%), and Scroll Depth (15%) as the top predictors of Health Score, and — to its credit — already flags that "the target itself is partly constructed from some of these inputs, so importance is descriptive rather than causal."

My methodology question: Where does the label come from, exactly? Health Score is explicitly defined as Impressions (30 pts) + Position (30 pts) + CTR (20 pts) + Scroll Depth (20 pts) — meaning 3 of the top-ranked "predictive" features (Position, Impressions, and by extension CTR) are literally components used to construct the label itself. This isn't a subtle leak; it's closer to asking a model to "predict" a sum using two of the numbers being summed. The paper's own caveat is honest and correct, but I'd push one step further: with this much direct construction overlap, does any meaningful signal remain in this importance ranking beyond "the label correlates with its own ingredients"? I'd want to see the same Random Forest re-run using only the features that are not part of the Health Score formula (Content Age, Word Count, Days Visible, AI Sessions) to see whether those carry real predictive signal — that would be a more honest test of what actually drives health, versus what's definitionally guaranteed to.

Finding B: Finding #8 — The Age-Freshness Matrix ("old content that gets refreshed performs nearly as well as new content")

The paper's own Methodology section states the ML/local-active-content sample is restricted to content with impressions_90d > 0 and sessions_90d > 0 — meaning a page must currently be earning some visibility to be included in this analysis at all.

My methodology question: Does the validation design actually carry this claim? The "365+ age, refreshed" quadrant (Health 44.6) being close to "31-90 age, fresh" (Health 44.1) is presented as evidence that "old content that gets refreshed performs nearly as well as new content." But by the sample's own inclusion rule, any old page that was refreshed and didn't recover (i.e., stayed at zero impressions after refresh) would be excluded from this comparison entirely — the sample can only ever show refreshed-old pages that already have some visibility. This is survivorship bias baked into the sampling frame, not just noise. The paper does flag this exact risk for the tiny 365+ × 361+ cell specifically, which is good practice — but I'd extend the same caution to the headline 365+ × 0-30 fresh comparison too, since it's built on the same active-content filter, just with a larger (and therefore easier to trust implicitly) sample size. A stronger validation design would report what fraction of refreshed old pages ever re-entered the active-content sample at all, alongside the health score of the ones that did.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

before_after = pd.DataFrame({
    'Stage': ['Week 4: Linear Regression, ungrouped random split', 'Week 5: Random Forest, grouped-by-client split'],
    'R2': [0.0033, -0.0074],
    'MAE': [None, 11.43]
})
print(before_after)

                                               Stage      R2    MAE
0  Week 4: Linear Regression, ungrouped random split  0.0033    NaN
1     Week 5: Random Forest, grouped-by-client split -0.0074  11.43


Week 4's Linear Regression scored R² = 0.0033 on an ungrouped random split — small, but positive. Week 5 re-ran the same feature family under a grouped-by-client split (Random Forest, R² = -0.0074, MAE = 11.43). The score got worse, not better, once client-level leakage was removed. This is the honest before/after: Week 4's small positive score was very likely inflated by rows from the same client appearing in both train and test, letting the model implicitly learn client-specific patterns rather than a real generalizable signal. The grouped split is the trustworthy number, even though it looks worse.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Final feature set: gsc_impressions, ga4_engaged_sessions, sessions_organic, sessions_ai, scroll_events")
print()
print("WITH deliberate leak (gsc_sum_position included), R²:", 0.0624)
print("WITHOUT leak (final honest feature set only), R²:", 0.0033)
print("Ratio: ~19x increase when the leak is present")


Final feature set: gsc_impressions, ga4_engaged_sessions, sessions_organic, sessions_ai, scroll_events

WITH deliberate leak (gsc_sum_position included), R²: 0.0624
WITHOUT leak (final honest feature set only), R²: 0.0033
Ratio: ~19x increase when the leak is present


The Week 3 leakage audit already tested this exact final feature set against a known leak source: adding gsc_sum_position (algebraically part of how gsc_avg_position is computed) raised R² from 0.0033 to 0.0624 — roughly 19x. This confirms the final 5-feature set is clean of that specific leak, since removing gsc_sum_position and gsc_clicks (both label components) restores the honest, much lower score. No other feature in the final set (gsc_impressions, ga4_engaged_sessions, sessions_organic, sessions_ai, scroll_events) is a mathematical component of the label, so the same class of leak does not apply to them. The audit does not rule out subtler, non-algebraic leakage (e.g., a feature correlating with the label through an unmeasured confound), which is a limitation to name honestly, not a gap to hide.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.original_claim = "Fixing rank on these pages should convert existing visibility into more clicks."
original_claim = "Fixing rank on these pages should convert existing visibility into more clicks."

rewritten_claim = "Content flagged as VISIBLE_BUT_POOR_RANK shows a measured, directional pattern — Signal 1 observed that CTR decreases as position worsens across this slice — which suggests ranking improvements are a plausible, decision-support lever for these pages, not a proven causal fix. The data does not establish that changing rank causes more clicks for any specific page; it only supports the association at the aggregate level."

print("ORIGINAL:", original_claim)
print()
print("REWRITTEN:", rewritten_claim)

ORIGINAL: Fixing rank on these pages should convert existing visibility into more clicks.

REWRITTEN: Content flagged as VISIBLE_BUT_POOR_RANK shows a measured, directional pattern — Signal 1 observed that CTR decreases as position worsens across this slice — which suggests ranking improvements are a plausible, decision-support lever for these pages, not a proven causal fix. The data does not establish that changing rank causes more clicks for any specific page; it only supports the association at the aggregate level.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.